# N03 - PSU-disjoint split + training-only preprocessing (plan stage 3)
Gate: no PSU overlap; fold counts and leakage assertions saved.

**What is frozen here** (plan 5.1-5.2, 4)
- One five-fold `StratifiedGroupKFold` on the whole cohort: groups = PSU (`v021`), stratify on stunting, shuffle, seed 42. **Fold 0 = final test partition** (fixed now, before any modelling - decision D13). Other four folds = development.
- Inside development: a second five-fold grouped split for model selection (same grouping, seed 42).
- For every development fold the preprocessing (medians, scaling, category levels, rare-category rule) is **fitted on that fold's training rows only**, then applied to its validation rows. The same fitted arrays are later used by *every* model, so all models see identical records.
- The test partition is **not** transformed or opened in this notebook.
- An inner PSU-disjoint early-stopping split (MLP only) is also frozen here **for the development CV folds**, so the validation fold is never used to stop training. (The final MLP does not use it - see N04.)
- Unknown-category policy: a category level that appears only in validation/test goes to the `__rare__` column if training had one, otherwise all dummy columns of that variable are 0. N03 counts how often that happens.

In [ ]:
# --- setup (same in every notebook) ---
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not on Colab - using local folder.")
import os, sys, time
import numpy as np
import pandas as pd
BASE_DIR = os.environ.get("THESIS_BASE_DIR", "/content/drive/MyDrive/THESIS_V2")
sys.path.append(BASE_DIR + "/code")
import bdhs_helpers as h
h.make_all_folders()
h.set_all_seeds(h.SEED_SPLIT)
h.print_environment("N03")

In [ ]:
h.require_signoff("n01_signoff.txt", "finish N01")
h.require_signoff("n02_signoff.txt", "finish N02 and freeze the predictors")

cohort = pd.read_csv(h.PRIVATE_DIR + "/cohort_private.csv")
frozen = pd.read_csv(h.OUTPUT_DIR + "/primary_predictors.csv")
predictors = []
for i in range(len(frozen)):
    if frozen["status"].iloc[i] == "primary":
        predictors.append(frozen["variable"].iloc[i])
print("predictors (", len(predictors), "):", predictors)

check_log = []
bad = []
for var in predictors:
    if var in h.FORBIDDEN_PREDICTORS:
        bad.append(var)
h.add_check(check_log, "no forbidden variable among predictors", len(bad) == 0, "forbidden found: " + str(bad))

y = cohort["stunted"].values.astype(int)
psu = cohort["v021"].values
row_ids = cohort["row_id"].values
decisions = h.load_coding_decisions()     # the written-down special-code decisions saved in N02
print(decisions[["variable", "code", "action"]].to_string(index=False))
clean = h.clean_predictors(cohort, predictors, decisions)
print("cohort rows:", len(y), "| PSUs:", len(np.unique(psu)))

### 1) Outer split: development vs test

In [ ]:
outer_fold = h.make_group_folds(y, psu, h.N_OUTER_FOLDS, h.SEED_SPLIT)
is_test = (outer_fold == h.TEST_FOLD)
is_dev = ~is_test

rows = []
for k in range(h.N_OUTER_FOLDS):
    mask = (outer_fold == k)
    rows.append([k, "test" if k == h.TEST_FOLD else "development", int(mask.sum()),
                 len(np.unique(psu[mask])), int(y[mask].sum()), round(float(y[mask].mean()), 4)])
outer_table = pd.DataFrame(rows, columns=["outer_fold", "role", "n_rows", "n_psu", "n_stunted", "prevalence"])
print(outer_table.to_string(index=False))
print("development rows:", int(is_dev.sum()), "| test rows:", int(is_test.sum()),
      "| test share of rows:", round(float(is_test.mean()), 3))

overlap = set(psu[is_dev]) & set(psu[is_test])
h.add_check(check_log, "no PSU in both development and test", len(overlap) == 0, "overlapping PSUs: " + str(len(overlap)))

### 2) Development cross-validation folds (inside development only)

In [ ]:
dev_index = np.where(is_dev)[0]
dev_fold_local = h.make_group_folds(y[dev_index], psu[dev_index], h.N_DEV_FOLDS, h.SEED_SPLIT)
dev_fold = np.full(len(y), -1, dtype=int)
dev_fold[dev_index] = dev_fold_local

rows = []
for k in range(h.N_DEV_FOLDS):
    val_mask = (dev_fold == k)
    train_mask = is_dev & (dev_fold != k)
    overlap = set(psu[train_mask]) & set(psu[val_mask])
    h.add_check(check_log, "dev fold " + str(k) + ": no PSU in both train and validation", len(overlap) == 0, "overlap: " + str(len(overlap)))
    rows.append([k, int(train_mask.sum()), int(val_mask.sum()), len(np.unique(psu[val_mask])),
                 int(y[val_mask].sum()), round(float(y[val_mask].mean()), 4)])
dev_table = pd.DataFrame(rows, columns=["dev_fold", "n_train", "n_val", "n_psu_val", "n_stunted_val", "prevalence_val"])
print(dev_table.to_string(index=False))

h.add_check(check_log, "test rows are in no development fold", bool((dev_fold[is_test] == -1).all()), "test rows with a dev fold: " + str(int((dev_fold[is_test] != -1).sum())))
h.add_check(check_log, "every development row is validated exactly once", int((dev_fold[is_dev] >= 0).sum()) == int(is_dev.sum()), "development rows: " + str(int(is_dev.sum())))

In [ ]:
# Save the frozen split manifest (row-level -> private) and its checksum (aggregate -> outputs)
partition = []
for k in outer_fold:
    partition.append("test" if k == h.TEST_FOLD else "dev")
manifest = pd.DataFrame({"row_id": row_ids, "v021": psu, "outer_fold": outer_fold,
                         "partition": partition, "dev_fold": dev_fold})
manifest_path = h.PRIVATE_DIR + "/split_manifest_private.csv"
h.save_table(manifest, manifest_path)
checksum = h.file_sha256(manifest_path)
f = open(h.OUTPUT_DIR + "/split_manifest_checksum.txt", "w")
f.write(checksum + "\n")
f.close()
print("manifest SHA-256:", checksum)
h.save_table(outer_table, h.OUTPUT_DIR + "/n03_outer_fold_counts.csv")
h.save_table(dev_table, h.OUTPUT_DIR + "/n03_dev_fold_counts.csv")

### 3) Fit preprocessing on training rows only - one fit per development fold
For each fold k: fit on the training rows -> apply to training and validation rows -> save arrays.
`X` = imputed, unscaled (for RF/XGBoost). `Xs` = numeric columns scaled with training mean/std (for LR/MLP). Same rows, same columns.

In [ ]:
demo_rows = []
unseen_rows = []
for k in range(h.N_DEV_FOLDS):
    val_mask = (dev_fold == k)
    train_mask = is_dev & (dev_fold != k)

    spec = h.fit_preprocessing(clean[train_mask], row_ids[train_mask], predictors)
    X_train, Xs_train = h.apply_preprocessing(clean[train_mask], spec, predictors)
    X_val, Xs_val = h.apply_preprocessing(clean[val_mask], spec, predictors)

    # leakage checks for this fold
    fit_hash_in_spec = spec["fit_ids_hash"].iloc[0]
    h.add_check(check_log, "fold " + str(k) + ": preprocessing fitted on exactly the training row ids",
                fit_hash_in_spec == h.ids_hash(row_ids[train_mask]), "hash " + fit_hash_in_spec)
    shared_ids = set(row_ids[train_mask]) & set(row_ids[val_mask])
    h.add_check(check_log, "fold " + str(k) + ": no validation row id in the training rows", len(shared_ids) == 0, "shared ids: " + str(len(shared_ids)))
    h.add_check(check_log, "fold " + str(k) + ": train/validation have the same encoded columns",
                list(X_train.columns) == list(X_val.columns), str(X_train.shape[1]) + " columns")

    unseen_val = h.count_unseen_cells(clean[val_mask], spec, predictors)
    unseen_rows.append([k, unseen_val])

    # inner early-stopping split (PSU-disjoint) inside the TRAINING rows
    es_mask = h.make_es_mask(y[train_mask], psu[train_mask], h.ES_SEED)
    psu_train = psu[train_mask]
    es_overlap = set(psu_train[es_mask]) & set(psu_train[~es_mask])
    h.add_check(check_log, "fold " + str(k) + ": early-stopping set shares no PSU with fitting rows", len(es_overlap) == 0, "overlap: " + str(len(es_overlap)))

    spec.to_csv(h.PRIVATE_DIR + "/preproc_spec_fold" + str(k) + ".csv", index=False)
    np.savez_compressed(h.PRIVATE_DIR + "/fold" + str(k) + "_arrays.npz",
                        train_ids=row_ids[train_mask], val_ids=row_ids[val_mask],
                        X_train=X_train.values, X_val=X_val.values,
                        Xs_train=Xs_train.values, Xs_val=Xs_val.values,
                        y_train=y[train_mask], y_val=y[val_mask], es_mask=es_mask,
                        feature_names=np.array(list(X_train.columns)))
    print("fold", k, "saved | train", X_train.shape, "val", X_val.shape, "| ES rows", int(es_mask.sum()))

    # demonstration: training-only statistics differ from full-development statistics
    if k == 0:
        for var in predictors:
            if var in h.NUMERIC_VARS:
                train_median = float(clean[var][train_mask].median())
                dev_median = float(clean[var][is_dev].median())
                demo_rows.append([var, round(train_median, 3), round(dev_median, 3)])

print()
print("Unknown-category policy: a validation level never seen in training goes to __rare__ if that column exists, otherwise all-zero.")
unseen_df = pd.DataFrame(unseen_rows, columns=["dev_fold", "validation_cells_all_zero_because_level_unseen"])
print(unseen_df.to_string(index=False))
h.save_table(unseen_df, h.OUTPUT_DIR + "/n03_unseen_level_counts.csv")

demo = pd.DataFrame(demo_rows, columns=["numeric_variable", "median_from_fold0_train", "median_from_all_dev"])
print()
print("Fold-0 imputation medians come from the fold-0 training rows only:")
print(demo.to_string(index=False))

### 4) Final development fit (used later to refit the final models on ALL development rows and to transform the locked test set)
No early-stopping rows are held out here: the final MLP is trained on every development row for a fixed epoch budget chosen in N04.

In [ ]:
spec_final = h.fit_preprocessing(clean[is_dev], row_ids[is_dev], predictors)
X_dev, Xs_dev = h.apply_preprocessing(clean[is_dev], spec_final, predictors)
h.add_check(check_log, "final fit used development rows only",
            spec_final["fit_ids_hash"].iloc[0] == h.ids_hash(row_ids[is_dev]) and len(set(row_ids[is_dev]) & set(row_ids[is_test])) == 0,
            "n_fit = " + str(int(is_dev.sum())))

spec_final.to_csv(h.PRIVATE_DIR + "/preproc_spec_final.csv", index=False)
np.savez_compressed(h.PRIVATE_DIR + "/final_dev_arrays.npz", dev_ids=row_ids[is_dev],
                    X_dev=X_dev.values, Xs_dev=Xs_dev.values, y_dev=y[is_dev],
                    feature_names=np.array(list(X_dev.columns)))

feature_map = h.make_feature_map(list(X_dev.columns))
h.save_table(feature_map, h.OUTPUT_DIR + "/feature_map.csv")
print(feature_map.groupby("source_variable").size().to_string())
print("Total encoded columns:", len(feature_map))

In [ ]:
check_df = pd.DataFrame(check_log, columns=["check", "result", "detail"])
h.save_table(check_df, h.OUTPUT_DIR + "/n03_leakage_checks.csv")
print("Checks passed:", int((check_df["result"] == "PASS").sum()), "of", len(check_df))
print("Gate: no PSU overlap + all checks PASS -> go to N04.")
print("The test partition (", int(is_test.sum()), "rows ) has NOT been transformed or scored.")